# 20 — O1-S-LB64 aggregate + SigLIP stage trace (DEV-only)

Notebook này chỉ chạy sau khi **cả Quang và Trọng notebook 19 đều COMPLETE**.
Nó fail-closed trên hai worker manifest, 90 marker và hash của mọi artifact, rồi:

1. gộp 12.982 source-frame sample của 219 target event;
2. chỉ giữ 71 target event thuộc đúng 63 DEV query đã khóa ở notebook 18;
3. embed query bằng **đúng SigLIP text tower production**, padding cố định 64 token;
4. so best sampled source với best BTC trong inclusive interval;
5. tính rank nếu source được chèn vào toàn bộ 176.707 vector BTC;
6. xuất per-event O1-S-LB64, stage trace, math ledger và measurement ledger.

Đây là **lower bound ở tầng SigLIP dense**. Interval dài hơn 64 frame không được gọi
full/best O1-S. Fusion, reranker, scorer và full OCR/caption/BGE parity vẫn `BLOCKED`.
Notebook không đọc held-out, không ghi `keyframes-btc`, không ingest hoặc sửa live
store. Nó chỉ publish báo cáo vào `keyframe-augmentation-research`.

### Cách chạy

1. Import notebook từ nhánh `feature/quang/storage`, chọn GPU và bật Internet.
2. Bật Kaggle Secrets `GITHUB_TOKEN`, `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`.
3. Không cần attach Dataset. Giữ nguyên `RUN_SCOPE="AGGREGATE"` và Save Version.
4. Thành công khi cuối log có `O1S_LB64_AGGREGATE_COMPLETE` và `events=71`.


## 0 — Khóa run; không có shard để đổi

In [ ]:
from pathlib import Path

RUN_SCOPE = "AGGREGATE"
CONFIRM_AGGREGATE = "O1S-LB64-DEV"
RUN_ID = "o1s-lb64-aggregate-v1"
SCAN_RUN_ID = "o1s-lb64-scan-v1"
RUNNERS = ("quang", "trong")
SAMPLING_POLICY = "inclusive_uniform_integer_lb64"
MAX_SAMPLES_PER_EVENT = 64
EXPOSURE_DEPTH = 100
SIGNAL_IMPROVEMENT_EPSILON = 1e-6

EXPECTED_SCAN_PLAN_HASH = "3d524cbfa77ceae0026a51ff48bdab0685fb9f66abca0975520112ae353e997c"
EXPECTED_SCAN_REPO_COMMIT = "1218f394671f1402d9d05dbcc79e3bb4ca4e926d"
EXPECTED_NB18_REPORT_SHA256 = "b089835a679809cfb1d2a1f79580343a8ceb0fe34846452038101f0db3a59228"
EXPECTED_TARGETS_SHA256 = "6929818a00205179f1d96c394f2930f43f88e9d536edccaf169b0632b5ee716b"
EXPECTED_SIGLIP_CANARY_REPORT_HASH = "45828774efcb0e291538b3405e41c8c720c9bb4e5c4e6588401c7a0547ca5a3c"
EXPECTED_TARGETS = 219
EXPECTED_SCAN_VIDEOS = 90
EXPECTED_SCAN_FRAMES = 12982
EXPECTED_DEV_QUERIES = 63
EXPECTED_DEV_TARGETS = 71
EXPECTED_BTC_SIGLIP_ROWS = 176707

CHECKPOINT = "google/siglip-so400m-patch14-384"
MODEL_REVISION = "9fdffc58afc957d1a03a25b10dba0329ab15c2a3"
SIGLIP_DIMENSION = 1152
TEXT_MAX_LENGTH = 64
TEXT_BATCH_SIZE = 32

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/o1s-lb64-aggregate")
REMOTE = WORK / "remote"
OUTPUT = WORK / "output"
for path in (WORK, REMOTE, OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

if RUN_SCOPE != "AGGREGATE" or CONFIRM_AGGREGATE != "O1S-LB64-DEV":
    raise RuntimeError("Aggregate lock is closed")


## 1 — Refresh repo, GPU compatibility và dependencies

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
from collections import Counter, defaultdict


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1800:]}")
    return result.stdout.strip()


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    github_token = secrets.get_secret("GITHUB_TOKEN")
    os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
    os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
except Exception as error:
    raise RuntimeError("Thiếu hoặc chưa bật quyền Kaggle Secrets GitHub/Modal") from error

askpass = WORK / "git-askpass.sh"
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO):
            raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=git_env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=git_env)
finally:
    askpass.unlink(missing_ok=True)


def torch_gpu_probe() -> dict:
    value = run([
        sys.executable,
        "-c",
        "import json,torch; c=torch.cuda.get_device_capability(0); "
        "print(json.dumps({'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),"
        "'capability':list(c),'arch_list':torch.cuda.get_arch_list()}))",
    ])
    return json.loads(value.splitlines()[-1])


probe_before = torch_gpu_probe()
required_arch = f"sm_{probe_before['capability'][0]}{probe_before['capability'][1]}"
if required_arch not in probe_before["arch_list"]:
    run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
        "torch==2.12.1", "torchvision==0.27.1",
        "--index-url", "https://download.pytorch.org/whl/cu126",
    ])
probe_after = torch_gpu_probe()
if required_arch not in probe_after["arch_list"]:
    raise RuntimeError(f"PyTorch không hỗ trợ GPU được cấp: {probe_after}")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "notebooks/requirements-kaggle-btc-images.txt")])
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import yaml
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

from mfe.contracts.ids import parse_keyframe_id
from mfe.core.config import load_config
from mfe.offline.augmentation.io import (
    canonical_hash,
    read_jsonl,
    sha256_file,
    write_json_atomic,
    write_jsonl_atomic,
)
from mfe.offline.augmentation.o1s_aggregation import (
    O1SEventTrace,
    adversarial_counterexample_summary,
    classify_siglip_failure_stage,
    cosine_scores,
    deterministic_best,
    deterministic_injected_rank,
    interval_candidate_indices,
    normalize_feature_rows,
    o1s_math_call_site_ledger,
    validate_o1s_worker_manifest,
)
from mfe.offline.augmentation.oracle_artifacts import OracleArtifactTarget
from mfe.offline.augmentation.representative_oracle import (
    RepresentativeResolvedSample,
    build_representative_scan_plan,
    partition_representative_scan,
    representative_scan_plan_hash,
    validate_scan_completion_marker,
)
from mfe.offline.bundle import BundleLayout, freeze_environment
from mfe.offline.source import load_catalogue
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.translate import build_translator

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 20 cần GPU: bật Accelerator GPU rồi Run All lại")

repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
producer_config_path = REPO / "config/keyframe_augmentation_kaggle.yaml"
producer_config = yaml.safe_load(producer_config_path.read_text(encoding="utf-8"))
btc = ModalVolumeTransport(producer_config["btc"]["volume"], create_if_missing=False, version=2)
research = ModalVolumeTransport(producer_config["output"]["volume"], create_if_missing=False, version=2)
btc_layout = BundleLayout(producer_config["btc"]["bundle"])
upstream_root = producer_config["output"]["root"]
scan_root = f"{upstream_root}/gate1-o1s-lb64-scan-v1"
nb18_root = f"{upstream_root}/gate1-oracle-eval-quang-v1"
output_root = f"{upstream_root}/gate1-o1s-lb64-aggregate-v1"
print({"repo_commit": repo_commit, "gpu": torch.cuda.get_device_name(0), "scope": RUN_SCOPE})


def load_npz_vectors(path: Path) -> tuple[list[str], np.ndarray]:
    with np.load(path, allow_pickle=False) as artifact:
        if set(artifact.files) != {"ids", "vectors"}:
            raise RuntimeError(f"Unexpected vector NPZ members: {artifact.files}")
        ids = [str(value) for value in artifact["ids"].tolist()]
        vectors = np.asarray(artifact["vectors"], dtype=np.float32)
    return ids, vectors


## 2 — Khóa notebook 18, scan plan và hai worker manifest

In [ ]:
nb18_report_path = REMOTE / "oracle-eval-report.json"
targets_path = REMOTE / "oracle-targets-resolved.jsonl"
research.get(f"{nb18_root}/oracle-eval-report.json", nb18_report_path)
research.get(f"{nb18_root}/oracle-targets-resolved.jsonl", targets_path)
if sha256_file(nb18_report_path) != EXPECTED_NB18_REPORT_SHA256:
    raise RuntimeError("Notebook 18 report SHA drift")
if sha256_file(targets_path) != EXPECTED_TARGETS_SHA256:
    raise RuntimeError("Oracle targets SHA drift")

nb18 = json.loads(nb18_report_path.read_text(encoding="utf-8"))
nb18_payload = dict(nb18)
nb18_claimed = nb18_payload.pop("report_payload_hash", None)
if nb18_claimed != canonical_hash(nb18_payload):
    raise RuntimeError("Notebook 18 report payload hash mismatch")
if nb18.get("status") != "ORACLE_DIAGNOSTIC_COMPLETE" or nb18.get("heldout_accessed") is not False:
    raise RuntimeError("Notebook 18 lineage/status invalid")

targets = [OracleArtifactTarget.model_validate(row) for row in read_jsonl(targets_path)]
if len(targets) != EXPECTED_TARGETS:
    raise RuntimeError(f"Expected {EXPECTED_TARGETS} targets, got {len(targets)}")
scan_plan = build_representative_scan_plan(targets, max_samples_per_event=MAX_SAMPLES_PER_EVENT)
if len(scan_plan) != EXPECTED_SCAN_FRAMES:
    raise RuntimeError("O1-S scan-frame count drift")
scan_plan_hash = representative_scan_plan_hash(scan_plan)
if scan_plan_hash != EXPECTED_SCAN_PLAN_HASH:
    raise RuntimeError("O1-S scan-plan hash drift")

catalogue_path = REPO / producer_config["source"]["catalogue"]
catalogue = load_catalogue(catalogue_path)
assignments = partition_representative_scan(
    scan_plan,
    RUNNERS,
    {video_id: catalogue[video_id].size for video_id in {row.video_id for row in scan_plan}},
)
by_runner_video = {}
for runner, rows in assignments.items():
    grouped = defaultdict(list)
    for row in rows:
        grouped[row.video_id].append(row)
    by_runner_video[runner] = dict(grouped)

worker_manifests = {}
for runner in RUNNERS:
    local = REMOTE / "worker-manifests" / f"{runner}.json"
    research.get(f"{scan_root}/worker-manifests/{runner}.json", local)
    manifest = json.loads(local.read_text(encoding="utf-8"))
    expected_videos = sorted(by_runner_video[runner])
    expected_frames = sum(len(rows) for rows in by_runner_video[runner].values())
    if not validate_o1s_worker_manifest(
        manifest,
        runner=runner,
        run_id=SCAN_RUN_ID,
        scan_plan_hash=scan_plan_hash,
        expected_assigned=expected_videos,
        expected_frame_count=expected_frames,
        expected_repo_commit=EXPECTED_SCAN_REPO_COMMIT,
        expected_notebook18_report_sha256=EXPECTED_NB18_REPORT_SHA256,
        expected_targets_sha256=EXPECTED_TARGETS_SHA256,
        expected_siglip_canary_report_hash=EXPECTED_SIGLIP_CANARY_REPORT_HASH,
    ):
        raise RuntimeError(f"{runner}: incomplete or invalid worker manifest")
    worker_manifests[runner] = manifest

assigned_videos = [video_id for runner in RUNNERS for video_id in worker_manifests[runner]["assigned"]]
if len(assigned_videos) != EXPECTED_SCAN_VIDEOS or len(set(assigned_videos)) != EXPECTED_SCAN_VIDEOS:
    raise RuntimeError("Worker assignments are not an exact disjoint 90-video set")
if sum(item["assigned_frame_count"] for item in worker_manifests.values()) != EXPECTED_SCAN_FRAMES:
    raise RuntimeError("Worker frame total is not 12,982")
print({"worker_status": {runner: "COMPLETE" for runner in RUNNERS}, "videos": len(assigned_videos), "frames": len(scan_plan)})


## 3 — Verify 90 marker + remote hashes và gộp source SigLIP

In [ ]:
def remote_paths(runner: str, video_id: str) -> dict[str, str]:
    return {
        "images": f"{scan_root}/images/{runner}/{video_id}.tar",
        "samples": f"{scan_root}/samples/{runner}/{video_id}.jsonl",
        "siglip": f"{scan_root}/siglip/{runner}/{video_id}.npz",
        "report": f"{scan_root}/reports/{runner}/{video_id}.json",
        "marker": f"{scan_root}/markers/{runner}/{video_id}.json",
    }


source_ids = []
source_vectors = []
sample_by_id = {}
source_artifact_hashes = {}
canary_hashes = set()
for runner in RUNNERS:
    for ordinal, video_id in enumerate(worker_manifests[runner]["assigned"], 1):
        paths = remote_paths(runner, video_id)
        marker_path = REMOTE / "markers" / runner / f"{video_id}.json"
        research.get(paths["marker"], marker_path)
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        observed = {path: research.sha256(path) for path in marker.get("artifacts", {})}
        if not validate_scan_completion_marker(
            marker,
            run_id=SCAN_RUN_ID,
            runner=runner,
            video_id=video_id,
            plan_hash=scan_plan_hash,
            observed_remote_hashes=observed,
        ):
            raise RuntimeError(f"{runner}/{video_id}: marker or remote artifact hash invalid")
        if set(marker["artifacts"]) != {paths["images"], paths["samples"], paths["siglip"], paths["report"]}:
            raise RuntimeError(f"{runner}/{video_id}: unexpected artifact set")

        samples_local = REMOTE / "samples" / runner / f"{video_id}.jsonl"
        siglip_local = REMOTE / "siglip" / runner / f"{video_id}.npz"
        report_local = REMOTE / "reports" / runner / f"{video_id}.json"
        for remote_path, local_path in (
            (paths["samples"], samples_local),
            (paths["siglip"], siglip_local),
            (paths["report"], report_local),
        ):
            research.get(remote_path, local_path)
            if sha256_file(local_path) != marker["artifacts"][remote_path]:
                raise RuntimeError(f"{runner}/{video_id}: downloaded artifact hash mismatch")
            source_artifact_hashes[remote_path] = marker["artifacts"][remote_path]

        samples = [RepresentativeResolvedSample.model_validate(row) for row in read_jsonl(samples_local)]
        expected_requests = by_runner_video[runner][video_id]
        if [sample.request for sample in samples] != expected_requests:
            raise RuntimeError(f"{runner}/{video_id}: sample requests drift from scan plan")
        ids, vectors = load_npz_vectors(siglip_local)
        ids = [str(value) for value in ids]
        vectors = np.asarray(vectors, dtype=np.float32)
        expected_ids = [sample.request.keyframe_id for sample in samples]
        if ids != expected_ids or vectors.shape != (len(ids), SIGLIP_DIMENSION):
            raise RuntimeError(f"{runner}/{video_id}: SigLIP identity/shape mismatch")
        if not np.isfinite(vectors).all():
            raise RuntimeError(f"{runner}/{video_id}: non-finite source vector")

        video_report = json.loads(report_local.read_text(encoding="utf-8"))
        report_payload = dict(video_report)
        report_claimed = report_payload.pop("report_payload_hash", None)
        if report_claimed != canonical_hash(report_payload):
            raise RuntimeError(f"{runner}/{video_id}: report payload hash mismatch")
        if video_report.get("status") != "O1S_LB64_VIDEO_COMPLETE":
            raise RuntimeError(f"{runner}/{video_id}: incomplete video report")
        canary_hashes.add(video_report["siglip"]["canary_report_hash"])

        for sample in samples:
            if sample.request.keyframe_id in sample_by_id:
                raise RuntimeError(f"duplicate source sample: {sample.request.keyframe_id}")
            sample_by_id[sample.request.keyframe_id] = sample
        source_ids.extend(ids)
        source_vectors.append(vectors)
        if ordinal % 10 == 0 or ordinal == len(worker_manifests[runner]["assigned"]):
            print(f"verified {runner} {ordinal}/{len(worker_manifests[runner]['assigned'])}", flush=True)

if len(source_ids) != EXPECTED_SCAN_FRAMES or len(set(source_ids)) != EXPECTED_SCAN_FRAMES:
    raise RuntimeError("Combined source scan is not 12,982 unique frames")
if len(canary_hashes) != 1:
    raise RuntimeError(f"SigLIP canary drift across scan artifacts: {canary_hashes}")
if canary_hashes != {EXPECTED_SIGLIP_CANARY_REPORT_HASH}:
    raise RuntimeError(f"SigLIP canary differs from the locked notebook 19 canary: {canary_hashes}")
source_matrix = normalize_feature_rows(np.concatenate(source_vectors, axis=0))
source_position = {keyframe_id: index for index, keyframe_id in enumerate(source_ids)}
print({"source_rows": len(source_ids), "dimension": source_matrix.shape[1], "remote_integrity": "PASS"})


## 4 — Tải BTC global SigLIP read-only và khóa DEV queryset

In [ ]:
base_siglip_path = WORK / "btc-siglip-vectors.npz"
btc.get(btc_layout.flat("siglip_vectors"), base_siglip_path)
expected_base_siglip_hash = nb18["input_hashes"]["base_siglip"]
if sha256_file(base_siglip_path) != expected_base_siglip_hash:
    raise RuntimeError("BTC global SigLIP hash differs from notebook 18 baseline")
btc_ids_raw, btc_vectors_raw = load_npz_vectors(base_siglip_path)
btc_ids = [str(value) for value in btc_ids_raw]
btc_matrix = normalize_feature_rows(np.asarray(btc_vectors_raw, dtype=np.float32))
if len(btc_ids) != EXPECTED_BTC_SIGLIP_ROWS or len(set(btc_ids)) != EXPECTED_BTC_SIGLIP_ROWS:
    raise RuntimeError("BTC global SigLIP identity count drift")
if btc_matrix.shape != (EXPECTED_BTC_SIGLIP_ROWS, SIGLIP_DIMENSION):
    raise RuntimeError(f"BTC global SigLIP shape drift: {btc_matrix.shape}")

queryset_path = REPO / "eval/queryset_dev.jsonl"
if sha256_file(queryset_path) != nb18["input_hashes"]["dev_queryset"]:
    raise RuntimeError("DEV queryset hash differs from notebook 18")
queries = read_jsonl(queryset_path)
query_by_id = {str(row["query_id"]): row for row in queries}
if len(query_by_id) != EXPECTED_DEV_QUERIES:
    raise RuntimeError(f"Expected {EXPECTED_DEV_QUERIES} DEV queries, got {len(query_by_id)}")
dev_targets = sorted(
    (target for target in targets if target.query_id in query_by_id),
    key=lambda target: (target.query_id, target.event_index),
)
if len(dev_targets) != EXPECTED_DEV_TARGETS:
    raise RuntimeError(f"Expected {EXPECTED_DEV_TARGETS} DEV target events, got {len(dev_targets)}")
print({"btc_rows": len(btc_ids), "dev_queries": len(query_by_id), "dev_target_events": len(dev_targets), "btc_mode": "READ_ONLY"})


## 5 — Query text contract, deterministic translation và SigLIP text tower

In [ ]:
def event_query_text(query: dict, event_index: int) -> str:
    if query["type"] == "TRAKE":
        texts = query.get("event_texts") or []
        if not 0 <= event_index < len(texts):
            raise RuntimeError(f"{query['query_id']}: missing TRAKE event text {event_index}")
        return str(texts[event_index])
    if query["type"] == "QA":
        return str(query.get("retrieval_text") or query["text"])
    return str(query["text"])


translation_root = WORK / "translation-root"
translation_cache_input = REPO / "eval/translations.json"
translation_cache_runtime = translation_root / "eval/translations.json"
translation_cache_runtime.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(translation_cache_input, translation_cache_runtime)
translation_cache_input_hash = sha256_file(translation_cache_input)
config = load_config(REPO / "config/btc_b1.yaml", repo_root=REPO)
translator = build_translator(config, repo_root=translation_root)
if translator is None:
    raise RuntimeError("Production translation contract is disabled")

translation_rows = []
event_text_en = {}
for target in dev_targets:
    key = (target.query_id, target.event_index)
    query = query_by_id[target.query_id]
    text_vi = event_query_text(query, target.event_index)
    translation = translator.translate(text_vi, manual=query.get("text_en"))
    if translation is None:
        raise RuntimeError(f"{target.query_id}/{target.event_index}: no English SigLIP query")
    event_text_en[key] = translation.text_en
    translation_rows.append({
        "query_id": target.query_id,
        "event_index": target.event_index,
        "text_vi": text_vi,
        "text_en": translation.text_en,
        "tier": translation.tier,
        "model": translation.model,
    })
translator.save_cache()
translation_cache_runtime_hash = sha256_file(translation_cache_runtime)
translations_path = write_jsonl_atomic(OUTPUT / "translations-used.jsonl", translation_rows)
translation_usage = dict(translator.usage)
del translator
gc.collect()

snapshot_dir = Path(snapshot_download(CHECKPOINT, revision=MODEL_REVISION))
model_file_hashes = {
    str(path.relative_to(snapshot_dir)): sha256_file(path)
    for path in sorted(snapshot_dir.rglob("*"))
    if path.is_file()
}
model_snapshot_hash = canonical_hash(model_file_hashes)
tokenizer = AutoTokenizer.from_pretrained(snapshot_dir, local_files_only=True)
model = AutoModel.from_pretrained(snapshot_dir, local_files_only=True).eval().cuda()

ordered_event_keys = [(target.query_id, target.event_index) for target in dev_targets]
query_vectors = []
for start in range(0, len(ordered_event_keys), TEXT_BATCH_SIZE):
    keys = ordered_event_keys[start : start + TEXT_BATCH_SIZE]
    batch = tokenizer(
        [event_text_en[key] for key in keys],
        padding="max_length",
        max_length=TEXT_MAX_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        output = model.get_text_features(**batch)
        pooled = getattr(output, "pooler_output", None)
        features = pooled if pooled is not None else output
    query_vectors.append(features.float().cpu().numpy())
query_matrix = normalize_feature_rows(np.concatenate(query_vectors, axis=0))
if query_matrix.shape != (EXPECTED_DEV_TARGETS, SIGLIP_DIMENSION):
    raise RuntimeError(f"SigLIP query matrix shape drift: {query_matrix.shape}")
del model, tokenizer, query_vectors, batch, output, pooled, features
gc.collect()
torch.cuda.empty_cache()
print({"translated_events": len(translation_rows), "translation_usage": translation_usage, "model_snapshot_hash": model_snapshot_hash})


## 6 — Đo best source/BTC, injected rank và observable failure stage

In [ ]:
# Một GPU matmul duy nhất cho toàn bộ 71 query-event × 176.707 BTC vectors.
with torch.inference_mode():
    btc_tensor = torch.from_numpy(btc_matrix).to("cuda")
    query_tensor = torch.from_numpy(query_matrix).to("cuda")
    btc_score_matrix = (query_tensor @ btc_tensor.T).float().cpu().numpy()
del btc_tensor, query_tensor
torch.cuda.empty_cache()

membership_positions = defaultdict(list)
for keyframe_id, sample in sample_by_id.items():
    position = source_position[keyframe_id]
    for membership in sample.request.memberships:
        membership_positions[(membership.query_id, membership.event_index)].append(position)

b0_scores = dict(zip(nb18["variants"]["B0"]["query_ids"], nb18["variants"]["B0"]["scores"], strict=True))
o1r_scores = dict(zip(nb18["variants"]["O1-R-FINAL"]["query_ids"], nb18["variants"]["O1-R-FINAL"]["scores"], strict=True))

event_results = []
stage_traces = []
best_source_candidates = []
for event_position, target in enumerate(dev_targets):
    event_key = (target.query_id, target.event_index)
    query = query_by_id[target.query_id]
    start_frame, end_frame = target.frame_range_start, target.frame_range_end
    global_scores = btc_score_matrix[event_position]
    query_vector = query_matrix[event_position]

    source_positions = sorted(set(membership_positions[event_key]))
    if not source_positions:
        raise RuntimeError(f"{event_key}: no source samples linked to target")
    local_source_ids = [source_ids[index] for index in source_positions]
    local_source_scores = cosine_scores(source_matrix[source_positions], query_vector)
    best_source_id, best_source_score, _ = deterministic_best(local_source_ids, local_source_scores)
    best_source_frame = parse_keyframe_id(best_source_id)[1]
    best_source_rank = deterministic_injected_rank(
        btc_ids,
        global_scores,
        candidate_id=best_source_id,
        candidate_score=best_source_score,
    )

    interval_positions = interval_candidate_indices(
        btc_ids,
        video_id=target.video_id,
        start_frame=start_frame,
        end_frame=end_frame,
    )
    if len(interval_positions):
        interval_ids = [btc_ids[index] for index in interval_positions]
        interval_scores = global_scores[interval_positions]
        best_btc_id, best_btc_score, _ = deterministic_best(interval_ids, interval_scores)
        best_btc_frame = parse_keyframe_id(best_btc_id)[1]
        best_btc_rank = deterministic_injected_rank(
            btc_ids,
            global_scores,
            candidate_id=best_btc_id,
            candidate_score=best_btc_score,
        )
        o1s_delta = best_source_score - best_btc_score
    else:
        best_btc_id = None
        best_btc_score = None
        best_btc_frame = None
        best_btc_rank = None
        o1s_delta = None

    # Global top BTC outside the target interval, ordered score desc then identity asc.
    global_order = np.lexsort((np.asarray(btc_ids), -global_scores))
    interval_set = set(interval_positions.tolist())
    outside_position = next(index for index in global_order if int(index) not in interval_set)
    top_outside_id = btc_ids[int(outside_position)]
    top_outside_score = float(global_scores[int(outside_position)])
    top_outside_rank = deterministic_injected_rank(
        btc_ids,
        global_scores,
        candidate_id=top_outside_id,
        candidate_score=top_outside_score,
    )

    failure_stage = classify_siglip_failure_stage(
        has_btc_in_interval=best_btc_id is not None,
        source_signal_delta=o1s_delta,
        best_source_rank=best_source_rank,
        best_btc_rank=best_btc_rank,
        exposure_depth=EXPOSURE_DEPTH,
        improvement_epsilon=SIGNAL_IMPROVEMENT_EPSILON,
    )
    full_scan = end_frame - start_frame + 1 <= MAX_SAMPLES_PER_EVENT
    source_stage_scores = {"siglip_dense": best_source_score, "fusion": None, "reranker": None, "scorer": None}
    btc_stage_scores = {"siglip_dense": best_btc_score, "fusion": None, "reranker": None, "scorer": None}
    provenance = {
        "scan_plan": scan_plan_hash,
        "notebook18_report": EXPECTED_NB18_REPORT_SHA256,
        "oracle_targets": EXPECTED_TARGETS_SHA256,
        "dev_queryset": nb18["input_hashes"]["dev_queryset"],
        "btc_siglip": expected_base_siglip_hash,
        "source_artifacts": canonical_hash(source_artifact_hashes),
        "model_snapshot": model_snapshot_hash,
        "translation_cache_runtime": translation_cache_runtime_hash,
    }
    result = O1SEventTrace(
        query_id=target.query_id,
        query_type=query["type"],
        event_index=target.event_index,
        video_id=target.video_id,
        gt_interval_inclusive=(start_frame, end_frame),
        sampling_policy=SAMPLING_POLICY,
        full_interval_scan=full_scan,
        best_btc_frame_idx=best_btc_frame,
        best_btc_rank=best_btc_rank,
        best_btc_score_by_stage=btc_stage_scores,
        top_btc_outside_interval_id=top_outside_id,
        top_btc_outside_interval_rank=top_outside_rank,
        top_btc_outside_interval_score=top_outside_score,
        best_source_frame_idx=best_source_frame,
        best_source_rank=best_source_rank,
        best_source_score_by_stage=source_stage_scores,
        o1_t_delta=None,
        o1_s_delta=o1s_delta,
        o1_r_delta=float(o1r_scores[target.query_id] - b0_scores[target.query_id]),
        failure_stage=failure_stage,
        failure_stage_status="MEASURED_SIGLIP_BOUNDARY_ONLY_DOWNSTREAM_BLOCKED",
        provenance_hashes=provenance,
    )
    event_results.append(result)
    best_source_candidates.append({
        "query_id": target.query_id,
        "event_index": target.event_index,
        "query_type": query["type"],
        "video_id": target.video_id,
        "keyframe_id": best_source_id,
        "frame_idx": best_source_frame,
        "siglip_score": best_source_score,
        "injected_rank": best_source_rank,
        "o1_s_delta": o1s_delta,
        "full_interval_scan": full_scan,
    })
    for stage, status in (
        ("siglip_dense", "MEASURED"),
        ("fusion", "BLOCKED_NO_STAGE_TRACE"),
        ("reranker", "BLOCKED_NO_STAGE_TRACE"),
        ("scorer", "BLOCKED_NO_O1S_OFFICIAL_RUN"),
    ):
        stage_traces.append({
            "query_id": target.query_id,
            "event_index": target.event_index,
            "stage": stage,
            "status": status,
            "best_source_score": source_stage_scores[stage],
            "best_btc_score": btc_stage_scores[stage],
            "best_source_rank": best_source_rank if stage == "siglip_dense" else None,
            "best_btc_rank": best_btc_rank if stage == "siglip_dense" else None,
            "top_btc_outside_interval_id": top_outside_id if stage == "siglip_dense" else None,
            "top_btc_outside_interval_rank": top_outside_rank if stage == "siglip_dense" else None,
            "top_btc_outside_interval_score": top_outside_score if stage == "siglip_dense" else None,
        })
    if (event_position + 1) % 10 == 0 or event_position + 1 == len(dev_targets):
        print(f"measured event {event_position + 1}/{len(dev_targets)}", flush=True)

events_path = write_jsonl_atomic(OUTPUT / "o1s-event-results.jsonl", event_results)
traces_path = write_jsonl_atomic(OUTPUT / "o1s-stage-traces.jsonl", stage_traces)
candidates_path = write_jsonl_atomic(OUTPUT / "o1s-best-source-candidates.jsonl", best_source_candidates)
print({"events": len(event_results), "stage_traces": len(stage_traces), "best_sources": len(best_source_candidates)})


## 7 — Aggregate report, math ledger và measured provenance

In [ ]:
deltas = [row.o1_s_delta for row in event_results if row.o1_s_delta is not None]
source_better = [value for value in deltas if value > SIGNAL_IMPROVEMENT_EPSILON]
rank_improved = [
    row for row in event_results
    if row.best_btc_rank is not None and row.best_source_rank < row.best_btc_rank
]
adversarial_summary = adversarial_counterexample_summary(
    event_results,
    exposure_depth=EXPOSURE_DEPTH,
    improvement_epsilon=SIGNAL_IMPROVEMENT_EPSILON,
)
math_ledger = o1s_math_call_site_ledger()
math_path = write_json_atomic(OUTPUT / "math-call-sites.json", {"schema_version": 1, "call_sites": math_ledger})

hypothesis_backlog = [
    {
        "proposal_id": "P-REP-01",
        "mechanism": "Use O1-S-positive source representatives to improve the frozen selector.",
        "target_failure_slice": "representation",
        "expected_primary_metric_gain": "UNVERIFIED_UNTIL_OFFICIAL_FINAL_SCORE_DEV_AB",
        "expected_cost": "medium_gpu_feature_rebuild_plus_dev_ab",
        "risks": ["sampled-source overfit", "crowding", "missing full feature parity"],
        "minimum_implementation": "factorized selector arm using only frozen O1-S-positive diagnostics",
        "control_arm": "B0",
        "treatment_arm": "B3_REPRESENTATIVE_V2",
        "fixed_factors": ["delta_budget", "feature_pipeline", "queryset", "scorer"],
        "falsification_rule": "reject if equal-budget DEV Final Score does not beat B0 and matched B4",
        "prerequisites": ["full feature parity", "crowding audit", "equal-budget proof"],
        "priority_score": {"expected_gain": 3, "evidence_strength": 3, "implementation_cost": 3, "regression_risk": 3},
        "status": "ready" if source_better else "rejected",
    },
    {
        "proposal_id": "P-EXP-01",
        "mechanism": "Increase or calibrate exposure for semantically stronger source frames below top-k.",
        "target_failure_slice": "retrieval",
        "expected_primary_metric_gain": "UNVERIFIED_UNTIL_FACTORIZED_EXPOSURE_AB",
        "expected_cost": "low_to_medium_dev_only",
        "risks": ["latency", "candidate crowding", "reranker dilution"],
        "minimum_implementation": "hold selector fixed and vary exposure depth only",
        "control_arm": "B3_FIXED_SELECTOR_DEPTH_100",
        "treatment_arm": "B3_FIXED_SELECTOR_CALIBRATED_DEPTH",
        "fixed_factors": ["selector", "delta_budget", "features", "fusion_weights"],
        "falsification_rule": "reject if target ranks improve but official DEV Final Score or latency gate fails",
        "prerequisites": ["representative source gain outside exposure", "cold/warm latency measurement"],
        "priority_score": {"expected_gain": 3, "evidence_strength": 3, "implementation_cost": 2, "regression_risk": 4},
        "status": "ready" if adversarial_summary["source_signal_gain_but_still_outside_exposure"] else "backlog",
    },
    {
        "proposal_id": "P-PARITY-01",
        "mechanism": "Rebuild OCR, caption, BGE and downstream fingerprints for candidate parity.",
        "target_failure_slice": "fusion",
        "expected_primary_metric_gain": "NO_GAIN_ASSUMED_PREREQUISITE_ONLY",
        "expected_cost": "high_gpu_and_storage",
        "risks": ["environment drift", "feature interaction", "large rebuild cost"],
        "minimum_implementation": "numeric canary and fingerprint for every production-consumed feature",
        "control_arm": "B0",
        "treatment_arm": "PARITY_COMPLETE_DELTA",
        "fixed_factors": ["queryset", "scorer", "delta identities"],
        "falsification_rule": "block A/B if any consumed feature or index fingerprint differs from BTC contract",
        "prerequisites": ["OCR/caption/BGE artifacts", "isolated snapshot index"],
        "priority_score": {"expected_gain": 2, "evidence_strength": 5, "implementation_cost": 5, "regression_risk": 2},
        "status": "blocked",
    },
]
hypothesis_path = write_json_atomic(
    OUTPUT / "hypothesis-backlog.json",
    {"schema_version": 1, "proposals": hypothesis_backlog},
)

dirty_after_clone = run(["git", "status", "--porcelain"], cwd=REPO).splitlines()
if dirty_after_clone:
    raise RuntimeError(f"Notebook 20 checkout became dirty: {dirty_after_clone[:10]}")
workspace_safety = {
    "schema_version": 1,
    "status": "PASS",
    "repo_ref": REPO_REF,
    "repo_commit": repo_commit,
    "repo_clean": True,
    "scope": "DEV_ONLY_O1S_LB64_AGGREGATE",
    "read_only_volumes": [producer_config["btc"]["volume"]],
    "research_volume_read_prefixes": [nb18_root, scan_root],
    "research_volume_write_prefix": output_root,
    "baseline_provenance": {
        "notebook18_report_sha256": EXPECTED_NB18_REPORT_SHA256,
        "oracle_targets_sha256": EXPECTED_TARGETS_SHA256,
        "scan_plan_hash": scan_plan_hash,
        "scan_repo_commit": EXPECTED_SCAN_REPO_COMMIT,
        "siglip_canary_report_hash": EXPECTED_SIGLIP_CANARY_REPORT_HASH,
        "worker_manifest_payload_hashes": {
            runner: worker_manifests[runner]["manifest_payload_hash"] for runner in RUNNERS
        },
    },
    "heldout_accessed": False,
    "btc_mutated": False,
    "live_store_mutated": False,
}
workspace_safety_path = write_json_atomic(OUTPUT / "workspace-safety-manifest.json", workspace_safety)

input_ledger = {
    "schema_version": 1,
    "command": "Kaggle Save Version / Run All: notebook 20 O1-S-LB64 aggregate",
    "split": "dev",
    "repo_commit": repo_commit,
    "input_hashes": {
        "notebook18_report": EXPECTED_NB18_REPORT_SHA256,
        "oracle_targets": EXPECTED_TARGETS_SHA256,
        "scan_plan": scan_plan_hash,
        "worker_manifests": canonical_hash({runner: worker_manifests[runner]["manifest_payload_hash"] for runner in RUNNERS}),
        "source_artifacts": canonical_hash(source_artifact_hashes),
        "btc_siglip": expected_base_siglip_hash,
        "dev_queryset": nb18["input_hashes"]["dev_queryset"],
        "translation_cache_input": translation_cache_input_hash,
        "translation_cache_runtime": translation_cache_runtime_hash,
        "model_snapshot": model_snapshot_hash,
        "workspace_safety_manifest": sha256_file(workspace_safety_path),
    },
    "config": {
        "sampling_policy": SAMPLING_POLICY,
        "max_samples_per_event": MAX_SAMPLES_PER_EVENT,
        "exposure_depth": EXPOSURE_DEPTH,
        "signal_improvement_epsilon": SIGNAL_IMPROVEMENT_EPSILON,
        "text_max_length": TEXT_MAX_LENGTH,
        "checkpoint": CHECKPOINT,
        "revision": MODEL_REVISION,
    },
    "heldout_accessed": False,
    "btc_access": "READ_ONLY",
    "btc_mutated": False,
}
input_ledger_path = write_json_atomic(OUTPUT / "input-ledger.json", input_ledger)

report = {
    "schema_version": 1,
    "status": "O1S_LB64_AGGREGATE_COMPLETE",
    "decision_authority": "diagnostic_only",
    "split": "dev",
    "run_id": RUN_ID,
    "scan_run_id": SCAN_RUN_ID,
    "events": len(event_results),
    "queries": len({row.query_id for row in event_results}),
    "scan_videos": EXPECTED_SCAN_VIDEOS,
    "scan_frames": EXPECTED_SCAN_FRAMES,
    "full_interval_events": sum(row.full_interval_scan for row in event_results),
    "sampled_interval_events": sum(not row.full_interval_scan for row in event_results),
    "events_without_interval_btc": sum(row.best_btc_frame_idx is None for row in event_results),
    "events_with_source_signal_gain": len(source_better),
    "events_with_rank_improvement": len(rank_improved),
    "mean_o1_s_siglip_delta_represented_events": float(np.mean(deltas)) if deltas else None,
    "median_o1_s_siglip_delta_represented_events": float(np.median(deltas)) if deltas else None,
    "max_o1_s_siglip_delta": float(max(deltas)) if deltas else None,
    "failure_stage_counts": dict(sorted(Counter(row.failure_stage for row in event_results).items())),
    "adversarial_counterexamples": adversarial_summary,
    "hypothesis_backlog_sha256": sha256_file(hypothesis_path),
    "oracle_status": {
        "O1-T": "MEASURED_TEMPORAL_COVERAGE_BY_NOTEBOOK_11_NO_PER_EVENT_RETRIEVAL_DELTA",
        "O1-S": "MEASURED_LOWER_BOUND_SIGLIP_DENSE_ONLY",
        "O1-R": "MEASURED_FINAL_CANDIDATE_FORCE_RANK_BY_NOTEBOOK_18",
    },
    "full_o1s_status": "BLOCKED_SAMPLED_INTERVALS_AND_MISSING_FULL_FEATURE_PARITY",
    "feature_parity_status": "MEASURED_SIGLIP_CANARY_ONLY_OCR_CAPTION_BGE_DOWNSTREAM_BLOCKED",
    "failure_stage_status": "PARTIAL_SIGLIP_DENSE_ONLY_FUSION_RERANKER_SCORER_BLOCKED",
    "official_score_status": "UNVERIFIED_NO_O1S_OFFICIAL_AB_RUN",
    "translation_usage": translation_usage,
    "model": {
        "checkpoint": CHECKPOINT,
        "revision": MODEL_REVISION,
        "snapshot_hash": model_snapshot_hash,
        "text_padding": "max_length_64",
    },
    "environment": freeze_environment(),
    "input_ledger_sha256": sha256_file(input_ledger_path),
    "math_ledger_sha256": sha256_file(math_path),
    "prohibited_inference": "O1-S-LB64 SigLIP signal cannot authorize production or establish official-score uplift.",
    "next": "Use best-source and stage traces to decide whether selector or exposure/parity work is justified before B0-B4 DEV A/B.",
    "heldout_accessed": False,
    "btc_mutated": False,
}
report["report_payload_hash"] = canonical_hash(report)
report_path = write_json_atomic(OUTPUT / "o1s-aggregate-report.json", report)

output_hashes = {
    path.name: sha256_file(path)
    for path in (
        events_path,
        traces_path,
        candidates_path,
        translations_path,
        math_path,
        hypothesis_path,
        workspace_safety_path,
        input_ledger_path,
        report_path,
    )
}
measurement_ledger = {
    "schema_version": 1,
    "evidence_label": "MEASURED",
    "command": input_ledger["command"],
    "input_hashes": input_ledger["input_hashes"],
    "config_hash": canonical_hash(input_ledger["config"]),
    "output_hashes": output_hashes,
    "heldout_accessed": False,
    "btc_mutated": False,
}
measurement_path = write_json_atomic(OUTPUT / "measurement-ledger.json", measurement_ledger)
output_hashes[measurement_path.name] = sha256_file(measurement_path)
manifest = {
    "schema_version": 1,
    "status": "COMPLETE",
    "run_id": RUN_ID,
    "report_payload_hash": report["report_payload_hash"],
    "output_hashes": output_hashes,
    "heldout_accessed": False,
    "btc_mutated": False,
}
manifest["manifest_payload_hash"] = canonical_hash(manifest)
manifest_path = write_json_atomic(OUTPUT / "COMPLETE.json", manifest)
print(json.dumps({
    "status": report["status"],
    "events": report["events"],
    "source_signal_gain": report["events_with_source_signal_gain"],
    "rank_improvement": report["events_with_rank_improvement"],
    "failure_stage_counts": report["failure_stage_counts"],
    "adversarial_counterexamples": report["adversarial_counterexamples"],
    "full_o1s_status": report["full_o1s_status"],
}, ensure_ascii=False, indent=2))


## 8 — Publish research-only outputs và verify remote

In [ ]:
uploads = [
    (events_path, f"{output_root}/o1s-event-results.jsonl"),
    (traces_path, f"{output_root}/o1s-stage-traces.jsonl"),
    (candidates_path, f"{output_root}/o1s-best-source-candidates.jsonl"),
    (translations_path, f"{output_root}/translations-used.jsonl"),
    (math_path, f"{output_root}/math-call-sites.json"),
    (hypothesis_path, f"{output_root}/hypothesis-backlog.json"),
    (workspace_safety_path, f"{output_root}/workspace-safety-manifest.json"),
    (input_ledger_path, f"{output_root}/input-ledger.json"),
    (report_path, f"{output_root}/o1s-aggregate-report.json"),
    (measurement_path, f"{output_root}/measurement-ledger.json"),
]
research.put_many(uploads)
for local, remote_path in uploads:
    if research.sha256(remote_path) != sha256_file(local):
        raise RuntimeError(f"Remote output hash mismatch: {remote_path}")

# COMPLETE marker luôn ghi sau cùng, chỉ sau khi mọi output đã verify.
manifest_remote = f"{output_root}/COMPLETE.json"
research.put(manifest_path, manifest_remote)
if research.sha256(manifest_remote) != sha256_file(manifest_path):
    raise RuntimeError("Remote COMPLETE marker hash mismatch")
print(json.dumps({
    "status": "O1S_LB64_AGGREGATE_COMPLETE",
    "events": len(event_results),
    "remote_output": output_root,
    "manifest_sha256": sha256_file(manifest_path),
    "btc_mutated": False,
    "heldout_accessed": False,
}, ensure_ascii=False, indent=2))


## Cách đọc kết quả

- `events_with_source_signal_gain > 0`: sampled source có representative headroom ở
  tầng SigLIP; chưa chứng minh official score tăng.
- `representation`: source tốt hơn BTC và đủ khả năng xuất hiện trong top-100.
- `retrieval`: ngay cả best sampled source/BTC vẫn nằm ngoài exposure depth.
- `temporal`: inclusive interval không có BTC frame.
- `none`: tầng SigLIP không cho thấy gap quan sát được; downstream vẫn chưa được loại trừ.
- Full O1-S, fusion/reranker/scorer và production Go/No-Go vẫn `BLOCKED` như report.
